In [ ]:
from IROS._pipeline_support import _handle_dirpaths

import numpy as np

from bloodmoon.io import simulation_files
from bloodmoon.mask import codedmask
from bloodmoon.optim import model_sky

import darksun as ds
from darksun.data import get_data, get_catalogue

In [ ]:
skyfield = "GalacticCenter"

#mask_FITS = "wfm_mask_summer2021.fits"
#data_FITS = "galctr_rxte-sax_mask_summer2021_infdet_2-50keV_1ks"

mask_FITS = "wfm_mask_NTHT_20250725.fits"
data_FITS = "galctr_rxte-sax_mask_050_1040x17_infdet_2-50keV_1ks"

#skyfield = "IROSDummy"
#data_FITS = "catalog_withCXB_1Crab_infdet_2-50keV_1ks"

N_TEST = "testing"

mask_path, simul_data, save_path = _handle_dirpaths(
    mask=mask_FITS,
    skyfield=skyfield,
    simul=data_FITS,
    run_name=N_TEST,
)

VIGNETTING = True
PSFY = False
UPX, UPY = 5, 1
wfm = codedmask(mask_path, UPX, UPY)

cam_a = "cam1a"
cam_b = "cam1b"
dataset = 'detected'

filepaths = simulation_files(simul_data)
sdlA = get_data(filepaths[cam_a][dataset])
catalogueA = get_catalogue(filepaths[cam_a]['sources'])

simul_sky_camA, _ = ds.load_sky(save_path + f"sky_SIMUL_CAM1A_TEST_{N_TEST}.fits")
simul_sky_camB, _ = ds.load_sky(save_path + f"sky_SIMUL_CAM1B_TEST_{N_TEST}.fits")
log_camA, log_camB = ds.load_database(save_path + f"IROS_sources_database_TEST_{N_TEST}.fits")

In [ ]:
import pandas as pd

pd.DataFrame(
    {
        log_camA.name: log_camA.log['ID'],
        log_camB.name: log_camB.log['ID'],
        "same": np.array(log_camA.log['ID']) == np.array(log_camB.log['ID']),
    }
)

In [ ]:
cropx, cropy = (
    int(wfm.specs["slit_deltax"] * UPX / wfm.specs["mask_deltax"] + 5),
    int(wfm.specs["slit_deltay"] * UPY / wfm.specs["mask_deltay"] + 5),
)
plot_IROS_efficiency(
    log=log_camA,
    true_sky=simul_sky_camA,
    crp=(cropy, cropx),
    camera=wfm,
    vignetting=VIGNETTING,
    psfy=PSFY,
)

In [ ]:
plot_thetas_residues(
    log=log_camA,
    sdl=sdlA,
    catalogue=catalogueA,
    camera=wfm,
)    
plot_fluences_residues(
    log=log_camA,
    sdl=sdlA,
    catalogue=catalogueA,
)
plot_fluence_residues_vs_theta(
    log=log_camA,
    sdl=sdlA,
    catalogue=catalogueA,
    camera=wfm,
)

In [ ]:
show_skyfield(log=log_camA, camera=wfm)
show_skyfield(log=log_camB, camera=wfm)